# EHR-M-GAN (Standard): Standalone Generation

This notebook is a **completely standalone** tool for generating synthetic clinical data. All necessary model architectures and utilities are included directly in this file.

## Setup Instructions

1.  **Files Required**: You only need to upload your trained checkpoint (e.g., `best_m3gan.pth`) and the metadata `clinical_scaler.pkl` (optional, for feature names).
2.  **Run**: Execute the cells in order to generate a synthetic patient.

### 1. Model Architecture and Utilities
The code below contains the Standard EHR-M-GAN architecture and normalization logic.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pickle
import os
import matplotlib.pyplot as plt
import seaborn as sns

# --- UTILITIES ---
def np_rounding(data, decimals=0):
    return np.around(data, decimals)

def renormlizer(data, max_min, min_val):
    return data * max_min + min_val

# --- NETWORKS ---
class VAE_Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, num_layers=3):
        super(VAE_Encoder, self).__init__()
        dropout_rate = 0.2 if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout_rate)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x_t, hidden_state):
        out, new_hidden = self.lstm(x_t, hidden_state)
        out_squeeze = out.squeeze(1)
        mu = torch.clamp(self.fc_mu(out_squeeze), min=-20.0, max=20.0)
        logvar = torch.clamp(self.fc_logvar(out_squeeze), min=-20.0, max=20.0)
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std
        return z, mu, logvar, new_hidden

class VAE_Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim, num_layers=3):
        super(VAE_Decoder, self).__init__()
        dropout_rate = 0.2 if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(latent_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout_rate)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, z_t, hidden_state):
        out, new_hidden = self.lstm(z_t, hidden_state)
        logits = self.fc_out(out.squeeze(1))
        return torch.sigmoid(logits), logits, new_hidden

class AutoregressiveVAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, enc_layers, dec_layers, time_steps):
        super(AutoregressiveVAE, self).__init__()
        self.time_steps, self.input_dim, self.hidden_dim = time_steps, input_dim, hidden_dim
        self.encoder = VAE_Encoder(input_dim * 2, hidden_dim, latent_dim, enc_layers)
        self.decoder = VAE_Decoder(latent_dim, hidden_dim, input_dim, dec_layers)

    def reconstruct_decoder(self, z_seq):
        batch_size = z_seq.size(0)
        dec_hidden = (torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=z_seq.device),
                      torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=z_seq.device))
        rec_list, logits_list = [], []
        for t in range(self.time_steps):
            rec_t, logits_t, dec_hidden = self.decoder(z_seq[:, t, :].unsqueeze(1), dec_hidden)
            rec_list.append(rec_t.unsqueeze(1)); logits_list.append(logits_t.unsqueeze(1))
        return torch.cat(rec_list, dim=1), torch.cat(logits_list, dim=1)

class BilateralLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(BilateralLSTMCell, self).__init__()
        self.linear = nn.Linear(input_dim + hidden_dim + hidden_dim, 4 * hidden_dim, bias=False)
    def forward(self, x, h_self, c_self, h_coupled):
        gates = self.linear(torch.cat([x, h_self, h_coupled], dim=1))
        i, f, o, c_ = gates.chunk(4, dim=1)
        c_next = torch.sigmoid(f) * c_self + torch.sigmoid(i) * torch.tanh(c_)
        h_next = torch.sigmoid(o) * torch.tanh(c_next)
        return h_next, c_next

class BilateralGenerator(nn.Module):
    def __init__(self, noise_dim, hidden_dim, latent_dim, num_layers=3):
        super(BilateralGenerator, self).__init__()
        self.num_layers, self.hidden_dim = num_layers, hidden_dim
        self.cells = nn.ModuleList([BilateralLSTMCell(noise_dim if i == 0 else hidden_dim, hidden_dim) for i in range(num_layers)])
        self.fc_out = nn.Linear(hidden_dim, latent_dim)

class JointGenerator(nn.Module):
    def __init__(self, c_noise_dim, d_noise_dim, hidden_dim, c_latent_dim, d_latent_dim, num_layers=3):
        super(JointGenerator, self).__init__()
        self.num_layers, self.hidden_dim = num_layers, hidden_dim
        self.c_gen = BilateralGenerator(c_noise_dim, hidden_dim, c_latent_dim, num_layers)
        self.d_gen = BilateralGenerator(d_noise_dim, hidden_dim, d_latent_dim, num_layers)

    def forward(self, noise_c, noise_d):
        batch_size, time_steps = noise_c.size(0), noise_c.size(1)
        c_h = [torch.zeros(batch_size, self.hidden_dim, device=noise_c.device) for _ in range(self.num_layers)]
        c_c = [torch.zeros(batch_size, self.hidden_dim, device=noise_c.device) for _ in range(self.num_layers)]
        d_h = [torch.zeros(batch_size, self.hidden_dim, device=noise_c.device) for _ in range(self.num_layers)]
        d_c = [torch.zeros(batch_size, self.hidden_dim, device=noise_c.device) for _ in range(self.num_layers)]
        z_c_list, z_d_list = [], []
        for t in range(time_steps):
            c_x = noise_c[:, t, :]
            for i in range(self.num_layers):
                c_h[i], c_c[i] = self.c_gen.cells[i](c_x, c_h[i], c_c[i], d_h[i])
                c_x = c_h[i]
            z_c_list.append(torch.sigmoid(self.c_gen.fc_out(c_x)).unsqueeze(1))
            d_x = noise_d[:, t, :]
            for i in range(self.num_layers):
                d_h[i], d_c[i] = self.d_gen.cells[i](d_x, d_h[i], d_c[i], c_h[i])
                d_x = d_h[i]
            z_d_list.append(torch.sigmoid(self.d_gen.fc_out(d_x)).unsqueeze(1))
        return torch.cat(z_c_list, dim=1), torch.cat(z_d_list, dim=1)

### 2. Configuration and Execution
Adjust the paths and run the cell to generate a patient.

In [ ]:
class Args:
    checkpoint = "best_m3gan.pth"
    scaler_path = "clinical_scaler.pkl"
    gen_num_units = 512
    gen_num_layers = 3
    enc_layers = 3
    dec_layers = 3
    time_steps = 24
    c_dim = 12
    d_dim = 10

args = Args()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def generate():
    if not os.path.exists(args.checkpoint):
        print(f"❌ Checkpoint not found at {args.checkpoint}. Please upload it.")
        return
        
    ckpt = torch.load(args.checkpoint, map_location=device)
    c_vae = AutoregressiveVAE(args.c_dim, args.gen_num_units, 25, args.enc_layers, args.dec_layers, args.time_steps).to(device)
    d_vae = AutoregressiveVAE(args.d_dim, args.gen_num_units, 25, args.enc_layers, args.dec_layers, args.time_steps).to(device)
    noise_dim = min(int(args.c_dim / 2), int(args.d_dim / 2))
    joint_gen = JointGenerator(noise_dim, noise_dim, args.gen_num_units, 25, 25, args.gen_num_layers).to(device)

    c_vae.load_state_dict(ckpt['c_vae'])
    d_vae.load_state_dict(ckpt['d_vae'])
    joint_gen.c_gen.load_state_dict(ckpt['c_gen'])
    joint_gen.d_gen.load_state_dict(ckpt['d_gen'])
    c_vae.eval(); d_vae.eval(); joint_gen.eval()

    with torch.no_grad():
        z_c, z_d = joint_gen(torch.randn(1, args.time_steps, noise_dim, device=device), 
                             torch.randn(1, args.time_steps, noise_dim, device=device))
        fake_c, _ = c_vae.reconstruct_decoder(z_c)
        fake_d, _ = d_vae.reconstruct_decoder(z_d)
    
    fake_c = fake_c.cpu().numpy()[0]
    fake_d = np_rounding(fake_d.cpu().numpy()[0])
    
    if os.path.exists(args.scaler_path):
        with open(args.scaler_path, 'rb') as f: scaler = pickle.load(f)
        fake_c = renormlizer(fake_c.reshape(1, args.time_steps, -1), scaler['maxes'] - scaler['mins'], scaler['mins'])[0]
        c_names, d_names = scaler['feature_names'], scaler['discrete_names']
    else:
        c_names, d_names = [f"C{i}" for i in range(args.c_dim)], [f"D{i}" for i in range(args.d_dim)]

    sns.set_style("whitegrid")
    fig, axes = plt.subplots(2, 1, figsize=(10, 8))
    for i in range(min(5, args.c_dim)): axes[0].plot(fake_c[:, i], label=c_names[i])
    axes[0].set_title("Synthetic Patient: Continuous Features"); axes[0].legend()
    sns.heatmap(fake_d.T[:15, :], ax=axes[1], cmap="Reds", yticklabels=d_names[:15])
    axes[1].set_title("Synthetic Patient: Discrete Features"); plt.tight_layout(); plt.show()

generate()